# Download and Archive QuantAQ MODULAIR Data

This notebook downloads observations from the QuantAQ Cloud API and writes
one CSV file per year to the local `data/` directory.

The notebook is intentionally limited to data acquisition and archiving.
Exploratory plots and resampling are handled separately in
`modulair_exploration.ipynb`.

The instrument is a QuantAQ MODULAIR™ mounted on the roof of the BBH building
at Northeastern Illinois University.


## Credentials

Store the QuantAQ API key in a local `.env` file:

```text
QUANTAQ_APIKEY=your_key_here
```

The `.env` file should be listed in `.gitignore` and should never be committed
to GitHub.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import quantaq
from dotenv import load_dotenv
from quantaq.utils import to_dataframe

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SERIAL_NUMBER = "MOD-00079"

POLLUTANTS = [
    "co",
    "co2",
    "no",
    "no2",
    "o3",
    "pm1",
    "pm10",
    "pm25",
]

# Change these values to control the archive update.
COMPLETE_YEARS = [2022, 2023, 2024, 2025]
CURRENT_YEAR = 2026

In [ ]:
load_dotenv()

QUANTAQ_APIKEY = os.environ["QUANTAQ_APIKEY"]

client = quantaq.QuantAQAPIClient(
    api_key=QUANTAQ_APIKEY,
)

## Optional short API test

This cell requests one day of data and is useful for checking credentials and
connectivity before starting a full-year download.


In [ ]:
# Example of how do fetch data for a date range (good for < 1 day). Not used below.
data = client.data.list(sn="MOD-00079", start="2024-01-01 00:00", stop="2024-01-01 23:59")
data = to_dataframe(data)

## Download one year

The API is queried one day at a time. Available daily records are concatenated,
rows without any pollutant measurements are removed, and the resulting table
is written to `data/quantaq_<year>.csv`.

Existing files are overwritten rather than appended, so rerunning the notebook
does not create duplicate records.


In [ ]:

def download_year(client, serial_number, year, *, data_dir=DATA_DIR):
    """Download one calendar year of QuantAQ data and save it as CSV."""

    daily_frames = []

    dates = pd.date_range(
        start=f"{year}-01-01",
        end=f"{year}-12-31",
    )

    for date in dates:
        print(f"Fetching {date.date()}...", end="\r")

        try:
            day = to_dataframe(
                client.data.bydate(
                    sn=serial_number,
                    date=str(date.date()),
                )
            )
        except Exception as error:
            print(f"\nError on {date.date()}: {error}")
            continue

        if not day.empty:
            daily_frames.append(day)

    if not daily_frames:
        raise RuntimeError(
            f"No data were downloaded for {serial_number} in {year}."
        )

    df = pd.concat(
        daily_frames,
        ignore_index=True,
    )

    available_pollutants = [
        pollutant
        for pollutant in POLLUTANTS
        if pollutant in df.columns
    ]

    if available_pollutants:
        df = df.dropna(
            subset=available_pollutants,
            how="all",
        )

    output_file = data_dir / f"quantaq_{year}.csv"

    df.to_csv(
        output_file,
        index=False,
    )

    print(
        f"\nSaved {len(df):,} rows to {output_file}"
    )

    return df


## Build or refresh the complete-year archive

Run this cell when the historical archive needs to be rebuilt. Comment out
years that do not need to be downloaded again.


In [ ]:

for year in COMPLETE_YEARS:
    download_year(
        client,
        SERIAL_NUMBER,
        year,
    )


## Update the current, incomplete year

This cell can be rerun periodically. It overwrites the current-year CSV with
all observations available through the day on which it is run.


In [ ]:

download_year(
    client,
    SERIAL_NUMBER,
    CURRENT_YEAR,
)
